# Stained-Glass Pipeline — Hatch Reimplementation

A pure-Python rewrite of the Grasshopper definition
`gh/v8/1234 image+dividing+extrusion+speckle.ghx`, using **procedural hatch tiling**
patterns as the geometry engine instead of the GH panel components.

**Pipeline (each GH stage → Python):**

| Grasshopper | Here |
|---|---|
| `Import Image` / `Read` | `PIL.Image` + panel boundary |
| `Diamond/Hexagon/Triangular/Quad Panels` + `Stream Gate` | **hatch tessellation** (`HATCHES[pattern]`) |
| `Mesh Closest Point` + `Mesh Eval` (×2 branches) | `sample_colours` (one branch) |
| `K-Means Clustering` | `sklearn.cluster.KMeans` |
| `Extrude` + `Offset Surface` + `Offset Curve` (frame) | `trimesh.creation.extrude_polygon` + inset/lead network |
| `Create Material` (transparency) | PBR `baseColorFactor` + `alphaMode='BLEND'` |
| `Speckle Geometry` → `Publish` | **GLB export** (consumed by the WebGPU renderer) |

The two duplicated GH colour branches collapse into one; the hard-wired 4-pattern
`Stream Gate` becomes an open `pattern` string — add a hatch function to add a pattern.

## 0. Setup

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np
from PIL import Image
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from shapely.affinity import rotate
from sklearn.cluster import KMeans
import trimesh
import matplotlib.pyplot as plt
%matplotlib inline

# locate repo root (folder containing 'assets') from wherever the notebook runs
BASE = Path.cwd()
while not (BASE / 'assets').exists() and BASE != BASE.parent:
    BASE = BASE.parent
OUT = Path.cwd() / 'output_hatch'
OUT.mkdir(exist_ok=True)
print('repo root:', BASE)

## 1. Parameters
These are the equivalents of the GH `Number Slider` / `Value List` / `Boolean Toggle` controls.

In [ ]:
CFG = dict(
    image_path = str(BASE / 'assets' / 'image library' / 'TEST_IMAGE.jpg'),
    panel_width  = 200.0,   # panel size in model units
    panel_height = 300.0,
    pattern      = 'diamond',  # rectangle | brick | diamond | hexagonal | triangle | pat
    tile_size    = 16.0,    # characteristic cell size (procedural patterns)
    pat_file     = str(BASE / 'pipeline' / 'patterns' / 'acad.pat'),  # used when pattern == 'pat'
    pat_name     = 'HEX',   # which pattern in the library: HEX, HONEY, TRIANG, SQUARE, NET, BRICK, ...
    pat_scale    = 100.0,   # .pat unit -> model units (HEX side is 0.125 -> ~12.5 model units)
    n_colors     = 8,       # KMeans palette size (GH 'amount of colors')
    lead_gap     = 1.2,     # half-width of lead came between cells (GH Offset Curve)
    glass_depth  = 4.0,     # glass extrusion thickness (GH Extrude / cell thickness)
    frame_height = 8.0,     # lead came height, taller than glass (GH Frame Height)
    glass_alpha  = 0.55,    # glass transparency (GH Create Material -> Transparency)
    lead_color   = (28, 28, 32),
    seed         = 42,
)
CFG['image_path']

## 2. Hatch tessellation
Each generator tiles `[0,W]×[0,H]` (with overflow) with closed polygons, then we clip to the
panel boundary. This replaces the GH panel components + `Stream Gate` multiplexer.

In [ ]:
import shapely

# Vectorized hatch generators (shapely 2.x array API) -> ndarray[Polygon] over [0,W]x[0,H].
def _quads(X, Y, s):
    X, Y = X.ravel(), Y.ravel()
    return np.stack([np.c_[X, Y], np.c_[X+s, Y], np.c_[X+s, Y+s], np.c_[X, Y+s]], axis=1)

def hatch_rectangle(W, H, s):
    X, Y = np.meshgrid(np.arange(-1, int(W//s)+2)*s, np.arange(-1, int(H//s)+2)*s)
    return shapely.polygons(_quads(X, Y, s))

def hatch_brick(W, H, s):
    bw, bh, rows = 2*s, s, []
    for j in range(-1, int(H//bh)+2):
        off = bw/2 if j % 2 else 0.0
        xs = np.arange(-1, int(W//bw)+3)*bw - off
        y = float(j*bh)
        rows.append(np.stack([np.c_[xs, np.full_like(xs, y)], np.c_[xs+bw, np.full_like(xs, y)],
                              np.c_[xs+bw, np.full_like(xs, y+bh)], np.c_[xs, np.full_like(xs, y+bh)]], axis=1))
    return shapely.polygons(np.concatenate(rows, axis=0))

def hatch_diamond(W, H, s):
    pad = 0.75*max(W, H)
    X, Y = np.meshgrid(np.arange(int(-pad//s)-1, int((W+pad)//s)+1)*s,
                       np.arange(int(-pad//s)-1, int((H+pad)//s)+1)*s)
    q = _quads(X, Y, s).astype(float)
    a = np.radians(45); R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    c = np.array([W/2, H/2])
    return shapely.polygons((q - c) @ R.T + c)               # square grid rotated 45 deg

def hatch_hexagon(W, H, s):
    w, vstep = np.sqrt(3)*s, 1.5*s
    C, Rr = np.meshgrid(np.arange(-1, int(W//w)+3, dtype=float), np.arange(-1, int(H//vstep)+3, dtype=float))
    C, Rr = C.ravel(), Rr.ravel()
    cx = C*w + np.where(Rr.astype(int) % 2 == 1, w/2, 0.0)
    cy = Rr*vstep
    ang = np.radians(60*np.arange(6)+90)
    X = cx[:, None] + s*np.cos(ang)[None, :]
    Y = cy[:, None] + s*np.sin(ang)[None, :]
    return shapely.polygons(np.stack([X, Y], axis=2))

def hatch_triangle(W, H, s):
    h = s*np.sqrt(3)/2
    J, I = np.meshgrid(np.arange(-1, int(H//h)+2, dtype=float), np.arange(-1, int(W//s)+3, dtype=float))
    J, I = J.ravel(), I.ravel()
    x, y0, y1 = I*s, J*h, (J+1)*h
    up = np.stack([np.c_[x, y0], np.c_[x+s, y0], np.c_[x+s/2, y1]], axis=1)
    dn = np.stack([np.c_[x+s/2, y1], np.c_[x+1.5*s, y1], np.c_[x+s, y0]], axis=1)
    return np.concatenate([shapely.polygons(up), shapely.polygons(dn)])

HATCHES = {'rectangle': hatch_rectangle, 'brick': hatch_brick, 'diamond': hatch_diamond,
           'hexagonal': hatch_hexagon, 'triangle': hatch_triangle}

## 2b. (Optional) AutoCAD `.pat` importer

Drive the tessellation from a **real** AutoCAD hatch-pattern library instead of a built-in
generator. A `.pat` is families of (optionally dashed) parallel lines; we draw the line
network, snap coincident nodes, and reconstruct closed cells with `shapely.polygonize`.

`pipeline/patterns/acad.pat` is the stock AutoCAD library (72 patterns, from the public
[netDxf](https://github.com/haplokuon/netDxf) repo). To use it, set `CFG['pattern'] = 'pat'`,
`CFG['pat_name']` to a pattern, and `CFG['pat_scale']` to size it.

Good *tiling* patterns (enclose cells): **HEX**, **HONEY**, **TRIANG**, **SQUARE**, **NET**, **BRICK**.
Pure-fill hatches (`ANSI31`, parallel lines) and mortar/outline patterns (`AR-HBONE`) don't
enclose cells and are skipped. Drop any other `.pat` into `patterns/` and point `pat_file` at it.

In [ ]:
import math
import shapely
from shapely.geometry import LineString
from shapely.ops import polygonize


def parse_pat(path, name=None):
    """Parse a .pat file into line families ('pens'). A library (e.g. acad.pat) holds many
    named patterns; pass `name` to pick one, else the file must contain a single pattern."""
    blocks, cur, body = {}, None, []
    for line in open(path, encoding='utf-8', errors='ignore'):
        t = line.split(';')[0].rstrip('\n')
        if t.strip().startswith('*'):
            if cur is not None:
                blocks[cur] = body
            cur, body = t.strip()[1:].split(',', 1)[0].strip(), []
        elif t.strip():
            body.append(t)
    if cur is not None:
        blocks[cur] = body
    if name is not None:
        chosen = blocks.get(name) or blocks.get(name.upper())
        if chosen is None:
            raise KeyError(f'pattern {name!r} not found; available: {sorted(blocks)[:10]}...')
    elif len(blocks) == 1:
        chosen = next(iter(blocks.values()))
    else:
        raise ValueError(f'{path} has {len(blocks)} patterns - set CFG["pat_name"]')
    pens = []
    for line in chosen:
        nums = [float(x) for x in line.split(',') if x.strip() != '']
        if len(nums) >= 5:
            pens.append(dict(angle=nums[0], x0=nums[1], y0=nums[2],
                             dx=nums[3], dy=nums[4], dashes=nums[5:]))
    return pens


def _dash(tA, tB, dashes):
    if not dashes:
        return [(tA, tB)]
    Lp = sum(abs(x) for x in dashes)
    if Lp <= 1e-9:
        return [(tA, tB)]
    t, out = math.floor(tA / Lp) * Lp, []
    while t < tB:
        for d in dashes:
            seg = abs(d) if abs(d) > 1e-9 else 1e-6
            if d > 0:
                a, b = max(t, tA), min(t + seg, tB)
                if b > a:
                    out.append((a, b))
            t += seg
            if t >= tB:
                break
    return out


def _clip(P, u, bbox):
    big = 1e6
    inter = LineString([P - big * u, P + big * u]).intersection(box(*bbox))
    if inter.is_empty or inter.geom_type != 'LineString':
        return None
    a, b = np.array(inter.coords[0]), np.array(inter.coords[-1])
    tA, tB = (a - P) @ u, (b - P) @ u
    return (min(tA, tB), max(tA, tB))


def _pen_segments(pen, bbox, scale):
    a = math.radians(pen['angle'])
    u, v = np.array([math.cos(a), math.sin(a)]), np.array([-math.sin(a), math.cos(a)])
    base = np.array([pen['x0'], pen['y0']]) * scale
    dx, dy = pen['dx'] * scale, pen['dy'] * scale
    dashes = [d * scale for d in pen['dashes']]
    if abs(dy) < 1e-9:
        dy = scale
    corners = np.array([[bbox[0], bbox[1]], [bbox[2], bbox[1]],
                        [bbox[2], bbox[3]], [bbox[0], bbox[3]]])
    perp = [(c - base) @ v for c in corners]
    i0, i1 = int(math.floor(min(perp) / dy)) - 1, int(math.ceil(max(perp) / dy)) + 1
    segs = []
    for i in range(i0, i1 + 1):
        Pi = base + i * dx * u + i * dy * v
        rng = _clip(Pi, u, bbox)
        if rng is None:
            continue
        for t0, t1 in _dash(rng[0], rng[1], dashes):
            A, B = Pi + t0 * u, Pi + t1 * u
            segs.append(LineString([(A[0], A[1]), (B[0], B[1])]))
    return segs


def tiles_from_pat(path, W, H, scale, name=None):
    """Draw a .pat line network over the panel and polygonize it into tile cells."""
    m = scale * 3 + 0.05 * max(W, H)              # overflow so edge cells close
    bbox = (-m, -m, W + m, H + m)
    segs = []
    for pen in parse_pat(path, name):
        segs += _pen_segments(pen, bbox, scale)
    merged = shapely.set_precision(unary_union(segs), max(scale * 1e-3, 1e-6))  # snap nodes
    return [p for p in polygonize(merged) if p.area < W * H * 0.2]


# quick check across a few real acad.pat patterns
_bnd = box(0, 0, CFG['panel_width'], CFG['panel_height'])
for _name, _sc in [('HEX', 100), ('TRIANG', 80), ('SQUARE', 90), ('BRICK', 45)]:
    _raw = tiles_from_pat(CFG['pat_file'], CFG['panel_width'], CFG['panel_height'], _sc, _name)
    _cells = sum(1 for _p in _raw if _p.intersection(_bnd).area > _sc * _sc * 0.01)
    print(f"{_name:8s} (scale {_sc:3d}) -> {_cells} cells")

In [ ]:
def generate_tiles(cfg):
    W, H, s = cfg['panel_width'], cfg['panel_height'], cfg['tile_size']
    boundary = box(0, 0, W, H)
    if cfg['pattern'] == 'pat':
        scale = cfg.get('pat_scale', s)
        raw = tiles_from_pat(cfg['pat_file'], W, H, scale, cfg.get('pat_name'))
        min_area = scale * scale * 0.01
        tiles = []
        for p in raw:                                          # modest count -> simple loop clip
            c = (p if p.is_valid else p.buffer(0)).intersection(boundary)
            for g in (c.geoms if hasattr(c, 'geoms') else [c]):
                if isinstance(g, Polygon) and g.area > min_area:
                    tiles.append(g)
        return tiles
    raw = shapely.make_valid(HATCHES[cfg['pattern']](W, H, s))  # vectorized below
    clipped = shapely.intersection(raw, boundary)
    parts = shapely.get_parts(clipped)
    parts = parts[shapely.get_type_id(parts) == 3]             # polygons only
    parts = parts[shapely.area(parts) > s * s * 0.02]
    return list(parts)

tiles = generate_tiles(CFG)
print(f"{CFG['pattern']}: {len(tiles)} tiles")

## 3. Sample image colours per cell
Sample the source image at each tile centroid — the single-branch equivalent of the two
GH `Mesh Closest Point` → `Mesh Eval` colour-sampling chains.

In [ ]:
def sample_colours(tiles, cfg):
    img = np.asarray(Image.open(cfg['image_path']).convert('RGB'))
    ih, iw, _ = img.shape
    W, H = cfg['panel_width'], cfg['panel_height']
    cents = shapely.centroid(np.array(tiles, dtype=object))     # vectorized centroids
    u = np.clip(shapely.get_x(cents) / W, 0, 1)
    v = np.clip(shapely.get_y(cents) / H, 0, 1)
    px = np.minimum((u * iw).astype(int), iw - 1)
    py = np.minimum(((1 - v) * ih).astype(int), ih - 1)         # flip Y (image row 0 is the top)
    return img[py, px].astype(float)

raw_cols = sample_colours(tiles, CFG)
raw_cols.shape

## 4. KMeans palette quantization
Reduce the sampled colours to an `n_colors` palette — the GH `K-Means Clustering` nodes.

In [ ]:
def quantize(cols, cfg):
    n_unique = len({tuple(c) for c in cols.astype(int)})
    k = max(1, min(cfg['n_colors'], n_unique))
    km = KMeans(n_clusters=k, n_init=4, random_state=cfg['seed']).fit(cols)
    palette = km.cluster_centers_.astype(int)
    return palette[km.labels_], palette, km.labels_

quant, palette, labels = quantize(raw_cols, CFG)

# show the palette
fig, ax = plt.subplots(figsize=(len(palette), 1))
ax.imshow([palette/255]); ax.set_xticks(range(len(palette))); ax.set_yticks([])
ax.set_title(f'{len(palette)}-colour palette'); plt.show()

## 5. 2D preview — the window seen flat

In [ ]:
def preview2d(tiles, quant, cfg):
    bg = tuple(c/255 for c in cfg['lead_color'])
    fig, ax = plt.subplots(figsize=(cfg['panel_width']/30, cfg['panel_height']/30))
    ax.set_facecolor(bg)
    for t, col in zip(tiles, quant):
        g = t.buffer(-cfg['lead_gap'])
        for gg in (g.geoms if hasattr(g, 'geoms') else [g]):
            if gg.is_empty:
                continue
            xs, ys = gg.exterior.xy
            ax.fill(xs, ys, color=tuple(c/255 for c in col))
    ax.set_xlim(0, cfg['panel_width']); ax.set_ylim(0, cfg['panel_height'])
    ax.set_aspect('equal'); ax.axis('off')
    fig.savefig(OUT/'preview.png', dpi=150, bbox_inches='tight', facecolor=bg)
    plt.show()

preview2d(tiles, quant, CFG)

## 6. Extrude glass + lead came → GLB
Glass cells are inset by `lead_gap` and extruded to `glass_depth` with a translucent PBR
material; the lead came is `panel − glass`, extruded taller. Exports a GLB that the
WebGPU rendering page already consumes.

In [ ]:
def _explode(geom):
    return geom.geoms if hasattr(geom, 'geoms') else [geom]

def _concat(parts):                                            # stack meshes, skip per-mesh reprocessing
    V, F, off = [], [], 0
    for m in parts:
        V.append(m.vertices); F.append(m.faces + off); off += len(m.vertices)
    return trimesh.Trimesh(np.vstack(V), np.vstack(F), process=False)

def build_scene(tiles, quant, cfg):
    scene = trimesh.Scene()
    groups, glass_insets = defaultdict(list), []
    insets = shapely.buffer(np.array(tiles, dtype=object), -cfg['lead_gap'])   # vectorized inset
    for inset, col in zip(insets, quant):
        for g in _explode(inset):
            if isinstance(g, Polygon) and not g.is_empty and g.area > 1e-6:
                groups[tuple(int(c) for c in col)].append(g)
                glass_insets.append(g)
    for col, polys in groups.items():
        parts = []
        for poly in polys:
            try:
                parts.append(trimesh.creation.extrude_polygon(poly, height=cfg['glass_depth'], process=False))
            except Exception:
                pass
        if not parts:
            continue
        mesh = _concat(parts)
        r, g, b = col
        mesh.visual = trimesh.visual.TextureVisuals(material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[r/255, g/255, b/255, cfg['glass_alpha']],
            metallicFactor=0.0, roughnessFactor=0.12, alphaMode='BLEND', doubleSided=True))
        scene.add_geometry(mesh, geom_name=f'glass_{r}_{g}_{b}')
    boundary = box(0, 0, cfg['panel_width'], cfg['panel_height'])
    lead_region = boundary.difference(shapely.union_all(glass_insets))
    lead_parts = []
    for g in _explode(lead_region):
        if isinstance(g, Polygon) and not g.is_empty and g.area > 1e-6:
            try:
                lead_parts.append(trimesh.creation.extrude_polygon(g, height=cfg['frame_height'], process=False))
            except Exception:
                pass
    if lead_parts:
        lead = _concat(lead_parts)
        lr, lg, lb = cfg['lead_color']
        lead.visual = trimesh.visual.TextureVisuals(material=trimesh.visual.material.PBRMaterial(
            baseColorFactor=[lr/255, lg/255, lb/255, 1.0], metallicFactor=0.55, roughnessFactor=0.45))
        scene.add_geometry(lead, geom_name='lead_came')
    return scene

scene = build_scene(tiles, quant, CFG)
scene.export(OUT/'stained_glass.glb')
print('geometries:', len(scene.geometry), '-> wrote', OUT/'stained_glass.glb')

Inline 3D thumbnail (static render of the scene):

In [ ]:
try:
    png = scene.save_image(resolution=(480, 640), visible=True)
    from IPython.display import Image as IPImage, display
    display(IPImage(png))
except Exception as e:
    print('offscreen render unavailable in this environment:', e)
    print('Open output_hatch/stained_glass.glb in the WebGPU renderer instead.')

## Notes
- **Swap pattern**: set `CFG['pattern']` to any key in `HATCHES`; add a `hatch_*` function for a new procedural pattern, **or** set `pattern='pat'` + `pat_name` to use any pattern from the bundled AutoCAD `acad.pat` library (72 patterns; see 2b).
- **Flat-panel assumption**: hatches are planar. For curved surfaces, map cell polygons through the surface UV (or use a tessellation plugin); see the earlier discussion.
- **Outputs**: `output_hatch/preview.png`, `stained_glass.glb` (renderer-ready), `stained_glass.3dm`.